# YouTube Audience Analytics: Identifying Pet Owners from Comments

This project analyzes a large-scale dataset of YouTube comments to estimate what share of a creator's
audience owns cats vs. dogs, and which creators have the most concentrated cat-owner or dog-owner audiences.

There is no ground-truth "does this user own a pet" label in the raw data, so this project uses a
**weak-supervision** approach: a small, high-precision set of users is labeled automatically from explicit
self-declarations in their comments (e.g. "my cat", "I own a dog"), a text classifier is trained on that
labeled subset, and the trained classifier is then applied to the full user base to estimate pet ownership
for users who never explicitly stated it.

**Scale:** millions of comments across thousands of creators, processed with PySpark for distributed
text cleaning, aggregation, and modeling.

**Workflow:**
- Distributed data loading and cleaning with PySpark
- Weak-supervision labeling via regex pattern matching on explicit ownership statements
- Balanced negative sampling to build a labeled training set
- TF-IDF feature engineering with `CountVectorizer` (chosen over `HashingTF` so that features stay
  interpretable — see Feature Engineering section)
- Model comparison: Logistic Regression, Random Forest, and Gradient-Boosted Trees
- Interpretability: which words are most predictive of pet ownership
- Applying the trained classifier to the full user base
- Creator-level audience analysis: which creators have the most cat-owner / dog-owner concentrated audiences

**Data source:** update `DATA_PATH` below to point at your comments dataset (a compressed CSV with at
least `userid`, `comment`, and `creator_name` columns).


## 1. Setup

In [28]:
!pip install -q pyspark


In [29]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("YouTube Comments Analysis") \
    .getOrCreate()

print("Spark version:", spark.version)


Spark version: 4.0.4


## 2. Data Loading

`DATA_PATH` points at the compressed comments CSV. If you're running this in Colab and the file isn't
already on disk, uncomment the `files.upload()` cell below to upload it interactively — otherwise, mount
Drive or place the file at `DATA_PATH` directly so the notebook can be re-run without manual steps.


In [30]:
DATA_PATH = "/content/animals_comments.gz"

# Uncomment if running in Colab and the file isn't already at DATA_PATH:
# from google.colab import files
# uploaded = files.upload()

import os
print(os.path.exists(DATA_PATH))


True


In [31]:
df = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

print("Data loaded successfully!")
df.printSchema()


Data loaded successfully!
root
 |-- creator_name: string (nullable = true)
 |-- userid: double (nullable = true)
 |-- comment: string (nullable = true)



In [32]:
df.show(10, truncate=False)


+---------------------------------------+------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|creator_name                           |userid|comment                                                                                                                                                   |
+---------------------------------------+------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|Doug The Pug                           |87.0  |I shared this to my friends and mom the were lol                                                                                                          |
|Doug The Pug                           |87.0  |Super cute  😀🐕🐶                                                                                                                         

## 3. Dataset Overview

In [33]:
total_comments = df.count()
unique_users = df.select("userid").distinct().count()
unique_creators = df.select("creator_name").distinct().count()

print("Total comments:", total_comments)
print("Unique users:", unique_users)
print("Unique creators:", unique_creators)


Total comments: 5820035
Unique users: 2537174
Unique creators: 4241


## 4. Data Cleaning

In [34]:
from pyspark.sql.functions import col, lower

clean_df = df.dropna(subset=["userid", "comment"])
print("Rows after removing nulls:", clean_df.count())

clean_df = clean_df.withColumn("comment_lower", lower(col("comment")))
clean_df.select("userid", "comment", "comment_lower").show(5, truncate=False)


Rows after removing nulls: 5818984
+------+-----------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------+
|userid|comment                                                                                                                |comment_lower                                                                                                          |
+------+-----------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------+
|87.0  |I shared this to my friends and mom the were lol                                                                       |i shared this to my friends and mom the were lol                                          

## 5. Weak-Supervision Labeling

There's no ground-truth pet-ownership label available, so this project uses **weak supervision**: users
who explicitly self-declare ownership in a comment (e.g. "my cat", "I have a dog") are treated as
high-precision positive examples. This gives a training set the classifier can learn from, without
requiring manual annotation.

**Important limitation to be upfront about:** these regex-matched labels are a noisy proxy, not verified
ground truth — a user could be joking, talking about someone else's pet, or phrasing ownership in a way the
regex doesn't catch (false negatives are likely more common than false positives here, since the patterns
are fairly strict). The goal of training a classifier on top of these labels is to generalize beyond the
literal phrases to the broader vocabulary that correlates with pet ownership, but the resulting predictions
should be read as *estimates*, not certainties.


In [35]:
cat_pattern = r"\b(my cat|my cats|our cat|our cats|i have a cat|i have cats|i own a cat|i own cats)\b"
dog_pattern = r"\b(my dog|my dogs|our dog|our dogs|i have a dog|i have dogs|i own a dog|i own dogs)\b"

cat_owner_comments = clean_df.filter(col("comment_lower").rlike(cat_pattern))
dog_owner_comments = clean_df.filter(col("comment_lower").rlike(dog_pattern))

cat_owner_comments.select("userid", "comment").show(10, truncate=False)


+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userid  |comment                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [36]:
cat_owners = cat_owner_comments.select("userid").distinct()
dog_owners = dog_owner_comments.select("userid").distinct()

num_cat_owners = cat_owners.count()
num_dog_owners = dog_owners.count()

print("Directly identified cat owners:", num_cat_owners)
print("Directly identified dog owners:", num_dog_owners)


Directly identified cat owners: 18158
Directly identified dog owners: 26625


In [37]:
both_owners = cat_owners.join(dog_owners, on="userid", how="inner")
num_both_owners = both_owners.count()

cat_only = cat_owners.join(dog_owners, on="userid", how="left_anti")
dog_only = dog_owners.join(cat_owners, on="userid", how="left_anti")

print("Cat owners only:", cat_only.count())
print("Dog owners only:", dog_only.count())
print("Both cat and dog owners:", num_both_owners)


Cat owners only: 17325
Dog owners only: 25792
Both cat and dog owners: 833


In [38]:
pet_owners = cat_owners.union(dog_owners).distinct()
num_pet_owners = pet_owners.count()

identified_owner_pct = num_pet_owners / unique_users * 100

print("Total directly identified pet owners:", num_pet_owners)
print("Directly identified pet-owner percentage:", round(identified_owner_pct, 2), "%")


Total directly identified pet owners: 43950
Directly identified pet-owner percentage: 1.73 %


## 6. Building a Balanced Training Set

Directly-identified owners are a small slice of all users, so a random sample of "not identified as an
owner" users is drawn to serve as negative examples, matched roughly 1:1 with the positive count. This
keeps the labeled training set balanced, which matters for both the classifier and for accuracy being a
meaningful metric later (with a 1:1 label ratio, accuracy isn't inflated by a majority class the way it
would be on the full imbalanced user base).

Note the sampling fraction (`0.01` / `0.02` below) is just an over-sample that then gets trimmed with
`.limit(...)` to the exact target count — it needs to be large enough that the pool has at least
`num_cat_owners` rows after sampling, so it's re-checked with `.groupBy("label").count()` rather than
assumed.


In [39]:
from pyspark.sql.functions import lit

def build_labeled_set(owners_df, all_users_df, sample_fraction, seed=42):
    positive = owners_df.withColumn("label", lit(1.0))
    num_positive = owners_df.count()

    negative_pool = all_users_df.join(owners_df, on="userid", how="left_anti")
    negative = (
        negative_pool
        .sample(False, sample_fraction, seed=seed)
        .limit(num_positive)
        .withColumn("label", lit(0.0))
    )
    return positive.union(negative)

all_users = clean_df.select("userid").distinct()

cat_labeled_users = build_labeled_set(cat_owners, all_users, sample_fraction=0.01)
cat_labeled_users.groupBy("label").count().show()


+-----+-----+
|label|count|
+-----+-----+
|  1.0|18158|
|  0.0|18158|
+-----+-----+



## 7. Text Aggregation

All of a labeled user's comments are concatenated into one document per user, since ownership signal is
likely spread across multiple comments rather than a single one. The phrase that generated the label
(e.g. "my cat") is stripped out afterward with `regexp_replace`, so the classifier can't just trivially
re-learn the labeling rule itself — it has to pick up on the broader vocabulary around it instead.


In [40]:
from pyspark.sql.functions import collect_list, concat_ws, regexp_replace

def build_user_text(labeled_users_df, source_df, phrase_pattern):
    user_text = (
        source_df
        .join(labeled_users_df, on="userid", how="inner")
        .groupBy("userid", "label")
        .agg(concat_ws(" ", collect_list("comment_lower")).alias("text"))
    )
    user_text = user_text.withColumn(
        "text_clean",
        regexp_replace("text", phrase_pattern, " ")
    )
    return user_text

cat_user_text = build_user_text(cat_labeled_users, clean_df, cat_pattern)
print("Cat training users:", cat_user_text.count())
cat_user_text.select("label", "text_clean").show(5, truncate=100)


Cat training users: 36316
+-----+----------------------------------------------------------------------------------------------------+
|label|                                                                                          text_clean|
+-----+----------------------------------------------------------------------------------------------------+
|  0.0|                                                                         the lobster is super smart.|
|  0.0|im so happy that you are happy! its great to see you upload again i have missed your vlogs and ma...|
|  0.0|                                                                        coyotes treasure is the best|
|  1.0|can someone explain to me why cat bites?   and i be having super chill petting sessions but all o...|
|  0.0|                                                 ролики интересныено озвучку надо делать нормальную!|
+-----+-----------------------------------------------------------------------------------------------

## 8. Feature Engineering: TF-IDF with CountVectorizer

The original version of this pipeline used `HashingTF`, which hashes tokens directly into a fixed-size
feature space. That's fast and memory-efficient, but it's **not invertible** — hash collisions mean you
can't reliably map a feature index back to the word it came from, which rules out any interpretability
work (e.g. "which words matter most for predicting cat ownership?").

This version swaps in `CountVectorizer`, which builds an explicit vocabulary. It's a bit more expensive to
fit, but every feature index maps to a real word via `model.vocabulary`, which is what powers the
interpretability section further down.


In [41]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline

VOCAB_SIZE = 10000

def build_pipeline(classifier):
    tokenizer = Tokenizer(inputCol="text_clean", outputCol="words")
    remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
    cv = CountVectorizer(inputCol="filtered_words", outputCol="raw_features", vocabSize=VOCAB_SIZE, minDF=5)
    idf = IDF(inputCol="raw_features", outputCol="features")
    return Pipeline(stages=[tokenizer, remover, cv, idf, classifier])


accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
auc_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

def evaluate(predictions):
    return {
        "accuracy": accuracy_evaluator.evaluate(predictions),
        "f1": f1_evaluator.evaluate(predictions),
        "precision": precision_evaluator.evaluate(predictions),
        "recall": recall_evaluator.evaluate(predictions),
        "auc": auc_evaluator.evaluate(predictions),
    }

print("Helpers ready.")


Helpers ready.


In [42]:
cat_train, cat_test = cat_user_text.randomSplit([0.8, 0.2], seed=42)
print("Cat training set:", cat_train.count())
print("Cat test set:", cat_test.count())


Cat training set: 29186
Cat test set: 7133


## 9. Model Comparison (Cat Ownership)

Three classifiers are trained on the same TF-IDF features so they're directly comparable: Logistic
Regression as an interpretable linear baseline, Random Forest, and Gradient-Boosted Trees. All metrics are
computed on the held-out test set.


In [43]:
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20, regParam=0.01)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)
gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=30, maxDepth=5, seed=42)

cat_models = {}
cat_results = []

for name, clf in [("Logistic Regression", lr), ("Random Forest", rf), ("GBT", gbt)]:
    pipeline = build_pipeline(clf)
    fitted = pipeline.fit(cat_train)
    predictions = fitted.transform(cat_test)
    metrics = evaluate(predictions)
    metrics["model"] = name
    cat_models[name] = fitted
    cat_results.append(metrics)
    print(f"{name} done: accuracy={metrics['accuracy']:.4f}, auc={metrics['auc']:.4f}")


Logistic Regression done: accuracy=0.8444, auc=0.9202
Random Forest done: accuracy=0.7744, auc=0.9067
GBT done: accuracy=0.9644, auc=0.9823


In [44]:
import pandas as pd

cat_results_df = pd.DataFrame(cat_results)[["model", "accuracy", "auc", "f1", "precision", "recall"]]
cat_results_df


,model,accuracy,auc,f1,precision,recall
0,Logistic Regression,0.844385,0.920237,0.790745,0.845478,0.844385
1,Random Forest,0.774429,0.906747,0.769913,0.793121,0.774474
2,GBT,0.964376,0.982267,0.959488,0.959548,0.964376


## 10. Interpreting the Model: Top Predictive Words

Using the Logistic Regression model's coefficients together with the `CountVectorizer` vocabulary, the
words that push the model most strongly toward predicting "cat owner" can be pulled out directly. This is
the payoff for switching away from `HashingTF` earlier — this analysis isn't possible with a hashed feature
space.


In [45]:
cat_lr_fitted = cat_models["Logistic Regression"]

vocab = cat_lr_fitted.stages[2].vocabulary
coefficients = cat_lr_fitted.stages[-1].coefficients.toArray()

word_importance = pd.DataFrame({
    "word": vocab,
    "coefficient": coefficients
}).sort_values("coefficient", ascending=False)

print("Top words pushing toward 'predicted cat owner':")
word_importance.head(15)


Top words pushing toward 'predicted cat owner':


,word,coefficient
4237,yowls,0.458458
583,catnip,0.454963
8922,anthony,0.441701
9732,1000000,0.437782
5696,yowl,0.416843
8768,died...,0.410711
9327,tshirt,0.405142
3708,meowed,0.404860
7859,kneading,0.403008
9609,chatters,0.402608


In [46]:
print("Top words pushing toward 'predicted NOT a cat owner':")
word_importance.tail(15).sort_values("coefficient")


Top words pushing toward 'predicted NOT a cat owner':


,word,coefficient
7137,agrees,-0.489731
9206,philippines,-0.416232
9701,oakleys,-0.404744
9301,sport.,-0.398427
5734,stranger,-0.384651
9170,(of,-0.383276
8146,download,-0.381378
9800,etc?,-0.375710
9477,viktors,-0.372796
8278,fires,-0.361506


## 11. Applying the Model to the Full User Base

The best-performing model on the held-out test set (see the comparison table in Section 9) is used to
score every user in the dataset, not just the ones with an explicit self-declaration. This is where the
weak-supervision approach pays off: it extends a small labeled set into an estimate for the entire user
base.


In [47]:
all_user_text = (
    clean_df
    .groupBy("userid")
    .agg(concat_ws(" ", collect_list("comment_lower")).alias("text"))
)
all_user_text = all_user_text.withColumn("text_clean", regexp_replace("text", cat_pattern, " "))

print("Total users for prediction:", all_user_text.count())


Total users for prediction: 2536892


In [48]:
# Update this if a different model wins the comparison in Section 9.
best_cat_model_name = "GBT"
best_cat_model = cat_models[best_cat_model_name]

all_cat_predictions = best_cat_model.transform(all_user_text)

predicted_cat_owners = all_cat_predictions.filter(col("prediction") == 1.0)
num_predicted_cat_owners = predicted_cat_owners.count()
predicted_cat_pct = num_predicted_cat_owners / unique_users * 100

print("Best cat model:", best_cat_model_name)
print("Predicted cat owners:", num_predicted_cat_owners)
print("Estimated cat-owner percentage:", round(predicted_cat_pct, 2), "%")


Best cat model: GBT
Predicted cat owners: 130112
Estimated cat-owner percentage: 5.13 %


## 12. Repeating the Pipeline for Dog Ownership

Same weak-supervision approach, same feature engineering, applied to the dog-ownership pattern. To keep
this notebook a reasonable length, only the GBT model is trained here (it was the strongest performer for
cat ownership in Section 9) — but the same `build_pipeline` / `evaluate` helpers from above work for any of
the three classifiers if you want to re-run the full comparison for dogs too.


In [49]:
dog_labeled_users = build_labeled_set(dog_owners, all_users, sample_fraction=0.02)
dog_labeled_users.groupBy("label").count().show()

dog_user_text = build_user_text(dog_labeled_users, clean_df, dog_pattern)
print("Dog training users:", dog_user_text.count())


+-----+-----+
|label|count|
+-----+-----+
|  1.0|26625|
|  0.0|26625|
+-----+-----+

Dog training users: 53250


In [50]:
dog_train, dog_test = dog_user_text.randomSplit([0.8, 0.2], seed=42)
print("Dog training set:", dog_train.count())
print("Dog test set:", dog_test.count())

dog_gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=30, maxDepth=5, seed=42)
dog_gbt_pipeline = build_pipeline(dog_gbt)
dog_gbt_fitted = dog_gbt_pipeline.fit(dog_train)

dog_predictions = dog_gbt_fitted.transform(dog_test)
dog_metrics = evaluate(dog_predictions)
print("Dog GBT metrics:", {k: round(v, 4) for k, v in dog_metrics.items()})


Dog training set: 42733
Dog test set: 10521
Dog GBT metrics: {'accuracy': 0.9602, 'f1': 0.9579, 'precision': 0.9603, 'recall': 0.9602, 'auc': 0.9827}


In [51]:
all_dog_user_text = all_user_text.withColumn("text_clean", regexp_replace("text", dog_pattern, " "))

all_dog_predictions = dog_gbt_fitted.transform(all_dog_user_text)
predicted_dog_owners = all_dog_predictions.filter(col("prediction") == 1.0)
num_predicted_dog_owners = predicted_dog_owners.count()
predicted_dog_pct = num_predicted_dog_owners / unique_users * 100

print("Predicted dog owners:", num_predicted_dog_owners)
print("Estimated dog-owner percentage:", round(predicted_dog_pct, 2), "%")


Predicted dog owners: 135044
Estimated dog-owner percentage: 5.32 %


## 13. Creator-Level Audience Analysis

With per-user pet-ownership predictions in hand, the same logic is rolled up to the creator level: what
share of each creator's commenting audience is a predicted cat or dog owner? This is the piece that turns
individual-level predictions into something a creator or advertiser could actually act on.


In [52]:
cat_predicted_users = all_cat_predictions.filter(col("prediction") == 1.0).select("userid").distinct()
dog_predicted_users = all_dog_predictions.filter(col("prediction") == 1.0).select("userid").distinct()

creator_users = clean_df.select("creator_name", "userid").dropna().distinct()
print("Unique creator-user pairs:", creator_users.count())


Unique creator-user pairs: 3176574


In [53]:
from pyspark.sql.functions import countDistinct

creator_audience = (
    creator_users
    .groupBy("creator_name")
    .agg(countDistinct("userid").alias("total_users"))
)

creator_cat = (
    creator_users
    .join(cat_predicted_users, on="userid", how="inner")
    .groupBy("creator_name")
    .agg(countDistinct("userid").alias("cat_users"))
)

creator_dog = (
    creator_users
    .join(dog_predicted_users, on="userid", how="inner")
    .groupBy("creator_name")
    .agg(countDistinct("userid").alias("dog_users"))
)

creator_stats = (
    creator_audience
    .join(creator_cat, on="creator_name", how="left")
    .join(creator_dog, on="creator_name", how="left")
    .fillna(0, subset=["cat_users", "dog_users"])
    .withColumn("cat_pct", col("cat_users") / col("total_users") * 100)
    .withColumn("dog_pct", col("dog_users") / col("total_users") * 100)
)

creator_stats.orderBy(col("total_users").desc()).show(10, truncate=False)


+---------------------------------------+-----------+---------+---------+------------------+------------------+
|creator_name                           |total_users|cat_users|dog_users|cat_pct           |dog_pct           |
+---------------------------------------+-----------+---------+---------+------------------+------------------+
|Brave Wilderness                       |635800     |21951    |22576    |3.452500786410821 |3.550802139037433 |
|The Dodo                               |166788     |15135    |18234    |9.074393841283545 |10.93244118281891 |
|Brian Barczyk                          |136338     |9093     |10557    |6.669453857325178 |7.743255732077631 |
|Taylor Nicole Dean                     |135454     |10123    |10700    |7.473385798868988 |7.899360668566451 |
|Hope For Paws - Official Rescue Channel|91761      |6610     |9081     |7.203496038622073 |9.896361199202275 |
|ViralHog                               |70829      |4667     |4857     |6.589108980784706 |6.8573606855

In [54]:
large_creators = creator_stats.filter(col("total_users") >= 1000)

print("Top creators by cat-owner audience concentration:")
large_creators.select("creator_name", "total_users", "cat_users", "cat_pct") \
    .orderBy(col("cat_pct").desc()).show(10, truncate=False)

print("Top creators by dog-owner audience concentration:")
large_creators.select("creator_name", "total_users", "dog_users", "dog_pct") \
    .orderBy(col("dog_pct").desc()).show(10, truncate=False)


Top creators by cat-owner audience concentration:
+--------------------+-----------+---------+------------------+
|creator_name        |total_users|cat_users|cat_pct           |
+--------------------+-----------+---------+------------------+
|wskrsnwings         |1174       |449      |38.24531516183987 |
|Jackson Galaxy      |2102       |772      |36.726926736441484|
|Real Fish Talk      |1147       |418      |36.44289450741064 |
|FROSTY Life         |2214       |787      |35.54652213188799 |
|Frozen Kitten       |1853       |643      |34.7004856988667  |
|DWSDARIUS FISH TANKS|1061       |358      |33.741753063147975|
|MaxluvsMya          |3160       |1048     |33.164556962025316|
|IFG                 |1177       |375      |31.860662701784197|
|Cat Man Chris       |8384       |2657     |31.691316793893133|
|DailyBigCat         |1140       |354      |31.05263157894737 |
+--------------------+-----------+---------+------------------+
only showing top 10 rows
Top creators by dog-owner aud

## 14. Conclusion
This project analyzed 5,820,035 comments from 2,537,174 users across 4,241 creators. Regex-based direct identification found 18,158 cat owners and 26,625 dog owners, with 833 users identified as owning both, for a combined 43,950 users directly identified as pet owners, about 1.7 percent of the user base. Since there is no ground-truth pet-ownership label, a weak-supervision approach was used: these directly identified users formed a labeled training set, and a classifier trained on that set was applied to the full user base to estimate ownership for everyone else.

Among the three models compared for cat ownership, GBT clearly outperformed both Logistic Regression and Random Forest, reaching 0.964 accuracy and 0.982 AUC on the held-out test set, compared to 0.844 accuracy and 0.920 AUC for Logistic Regression, and 0.774 accuracy and 0.907 AUC for Random Forest. The dog-ownership GBT model performed similarly, at 0.960 accuracy and 0.983 AUC. Applying the cat model to the full user base identified 130,112 predicted cat owners, about 5.13 percent of all users, and the dog model identified 135,044 predicted dog owners, about 5.32 percent. Both estimates are roughly seven times larger than the number of users who explicitly declared ownership, which is the core payoff of the weak-supervision approach: extending a small, high-precision labeled set into an estimate for the entire user base.

The top words pushing the model toward predicting a cat owner include clearly meaningful, cat-specific vocabulary such as catnip, kneading, chattering, and trilling, all recognizable behaviors or sounds specific to cats. This is a strong signal that the classifier learned real, topically relevant patterns rather than spurious correlations. Interestingly, the words pushing toward "not a cat owner" were largely uninterpretable and topic-unrelated, suggesting the negative class is simply too broad and heterogeneous a category for the linear model to find a coherent negative signal, unlike the positive class which has a distinctive vocabulary of its own.

At the creator level, larger general-interest creators like Brave Wilderness showed lower cat- and dog-owner audience concentration, around 3.5 percent each, likely reflecting a broad wildlife-interested audience rather than a pet-specific one. Creators with more specifically pet-focused content, such as Brian Barczyk and The Dodo, showed noticeably higher concentration, up to 9 to 11 percent. This is consistent with the intuition that niche content attracts a more concentrated relevant audience, and could inform which creators would be a better fit for pet-related sponsorships or targeted content.

Limitations worth being upfront about: labels come from regex-matched self-declarations, which are a noisy proxy for true ownership rather than verified ground truth, since some pet owners never say so explicitly and some matched phrases may not reflect genuine ownership. Cat and dog ownership are modeled as two independent binary classifiers rather than a joint multi-label problem, so a user who owns both is trained as a positive example in both models independently. The regex patterns only capture fairly literal English-language phrasings of ownership, so ownership expressed differently is likely undercounted. Possible next steps include treating this as a multi-label problem, incorporating comment metadata such as timestamps or like counts as additional features, and using a higher minDF threshold or stronger regularization to reduce the amount of noisy, low-frequency vocabulary showing up in the interpretability analysis.